# Import

In [1]:
import os
import random

import pandas as pd
import numpy as np

from PIL import Image
from tqdm import tqdm 

from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.models as models
import torchvision.transforms as transforms
import torch.nn.functional as F
from torch import nn, optim

from sklearn.metrics import log_loss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


# Hyperparameter Setting

In [2]:
# CFG = {
#     'IMG_SIZE': 224,
#     'BATCH_SIZE': 64,
#     'EPOCHS': 10,
#     'LEARNING_RATE': 1e-4,
#     'SEED' : 42
# }

CFG = {
    'IMG_SIZE': 256,           # ✅ 더 디테일 반영
    'BATCH_SIZE': 32,          # ✅ 이미지 크기 증가에 따른 조정
    'EPOCHS': 50,              # ✅ 충분한 학습 시간 확보
    'LEARNING_RATE': 1e-3,     # ✅ 빠른 수렴 → scheduler로 줄이기 추천
    'SEED': 42,                # ✅ 재현성 확보
    'EARLY_STOPPING_PATIENCE': 5,  # ✅ 성능 고정 시 조기 종료
    'MODEL_NAME': 'resnet18',      # ✅ 향후 모델 변경 대비
    'NUM_CLASSES': 396,           # ✅ 클래스 수
    'T_0': 5,                     # ✅ CosineAnnealingWarmRestarts 초기 주기
    'T_MULT': 2,                  # ✅ CosineAnnealingWarmRestarts 배수 증가
    'WEIGHT_DECAY': 1e-4          # ✅ 일반화 향상을 위한 weight decay
}

# Fixed RandomSeed

In [3]:
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.cuda.manual_seed_all(seed)

seed_everything(CFG['SEED']) # Seed 고정

# CustomDataset

In [4]:
class CustomImageDataset(Dataset):
    def __init__(self, root_dir, transform=None, is_test=False):
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test
        self.samples = []

        if is_test:
            # 테스트셋: 라벨 없이 이미지 경로만 저장
            for fname in sorted(os.listdir(root_dir)):
                if fname.lower().endswith(('.jpg')):
                    img_path = os.path.join(root_dir, fname)
                    self.samples.append((img_path,))
        else:
            # 학습셋: 클래스별 폴더 구조에서 라벨 추출
            self.classes = sorted(os.listdir(root_dir))
            self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

            for cls_name in self.classes:
                cls_folder = os.path.join(root_dir, cls_name)
                for fname in os.listdir(cls_folder):
                    if fname.lower().endswith(('.jpg')):
                        img_path = os.path.join(cls_folder, fname)
                        label = self.class_to_idx[cls_name]
                        self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        if self.is_test:
            img_path = self.samples[idx][0]
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image
        else:
            img_path, label = self.samples[idx]
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label


# Data Load

In [5]:
# !unzip -q /kaggle/input/hecto-ai.zip -d /kaggle/working/
# train_root = '/kaggle/working/hecto-ai/train'

In [6]:
# import os

# for dirname, _, filenames in os.walk('/kaggle/input'):
#     print(dirname)

In [7]:
train_root = '/kaggle/input/train'
test_root = '/kaggle/input/test'

In [8]:
# train_transform = transforms.Compose([
#     transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                          std=[0.229, 0.224, 0.225])
# ])

# val_transform = transforms.Compose([
#     transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                          std=[0.229, 0.224, 0.225])
# ])

train_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'] + 32, CFG['IMG_SIZE'] + 32)),  # 여유롭게 크게 resize
    transforms.RandomResizedCrop(CFG['IMG_SIZE'], scale=(0.8, 1.0), ratio=(0.9, 1.1)),  # scale 살짝 확대, 비율 추가
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),  # hue 살짝 낮춤 → 색 왜곡 줄이기
    transforms.RandomRotation(degrees=7),  # 10도 → 7도로 약간 보수적으로
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),  # 📌 추가: 3D 왜곡에 강해짐
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [9]:
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

# 전체 데이터셋 로드
full_dataset = CustomImageDataset(train_root, transform=None)
print(f"총 이미지 수: {len(full_dataset)}")

# 라벨 리스트 추출
targets = [label for _, label in full_dataset.samples]
class_names = full_dataset.classes  # 🔍 class index → name 대응 위해

# Stratified Split
train_idx, val_idx = train_test_split(
    range(len(targets)),
    test_size=0.2,
    stratify=targets,
    random_state=CFG['SEED']  # 💡 seed 일치!
)

# transform 각각 적용된 Subset 생성
train_dataset = Subset(CustomImageDataset(train_root, transform=train_transform), train_idx)
val_dataset = Subset(CustomImageDataset(train_root, transform=val_transform), val_idx)
print(f"train 이미지 수: {len(train_dataset)}, valid 이미지 수: {len(val_dataset)}")

# DataLoader 정의 (worker 수 + memory pinning 추가)
train_loader = DataLoader(train_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False, num_workers=4, pin_memory=True)

총 이미지 수: 33137
train 이미지 수: 26509, valid 이미지 수: 6628


# Model Define

In [12]:
# class BaseModel(nn.Module):
#     def __init__(self, num_classes):
#         super(BaseModel, self).__init__()
#         self.backbone = models.resnet18(pretrained=False)  # ResNet18 모델 불러오기
#         self.feature_dim = self.backbone.fc.in_features 
#         self.backbone.fc = nn.Identity()  # feature extractor로만 사용
#         self.head = nn.Linear(self.feature_dim, num_classes)  # 분류기

#     def forward(self, x):
#         x = self.backbone(x)       
#         x = self.head(x) 
#         return x

class BaseModel(nn.Module):
    def __init__(self, num_classes):
        super(BaseModel, self).__init__()
        self.backbone = models.resnet18(weights=None)  # ✅ 사전학습 가중치 사용
        self.feature_dim = self.backbone.fc.in_features 
        self.backbone.fc = nn.Identity()  # feature extractor로만 사용

        self.dropout = nn.Dropout(p=0.5)  # ✅ Dropout 비율 증가 (50epoch 대비)
        self.bn = nn.BatchNorm1d(self.feature_dim)  # ✅ 일반화 성능 향상

        self.head = nn.Linear(self.feature_dim, num_classes)

    def forward(self, x):
        x = self.backbone(x)
        x = self.dropout(x)
        x = self.bn(x)
        x = self.head(x)
        return x

# Train/ Validation

In [13]:
# model = BaseModel(num_classes=len(class_names)).to(device)
# best_logloss = float('inf')

# # 손실 함수
# criterion = nn.CrossEntropyLoss()

# # 옵티마이저
# optimizer = optim.Adam(model.parameters(), lr=CFG['LEARNING_RATE'])

# # 학습 및 검증 루프
# for epoch in range(CFG['EPOCHS']):
#     # Train
#     model.train()
#     train_loss = 0.0
#     for images, labels in tqdm(train_loader, desc=f"[Epoch {epoch+1}/{CFG['EPOCHS']}] Training"):
#         images, labels = images.to(device), labels.to(device)
#         optimizer.zero_grad()
#         outputs = model(images)  # logits
#         loss = criterion(outputs, labels)
#         loss.backward()
#         optimizer.step()
#         train_loss += loss.item()

#     avg_train_loss = train_loss / len(train_loader)

#     # Validation
#     model.eval()
#     val_loss = 0.0
#     correct = 0
#     total = 0
#     all_probs = []
#     all_labels = []

#     with torch.no_grad():
#         for images, labels in tqdm(val_loader, desc=f"[Epoch {epoch+1}/{CFG['EPOCHS']}] Validation"):
#             images, labels = images.to(device), labels.to(device)
#             outputs = model(images)
#             loss = criterion(outputs, labels)
#             val_loss += loss.item()

#             # Accuracy
#             _, preds = torch.max(outputs, 1)
#             correct += (preds == labels).sum().item()
#             total += labels.size(0)

#             # LogLoss
#             probs = F.softmax(outputs, dim=1)
#             all_probs.extend(probs.cpu().numpy())
#             all_labels.extend(labels.cpu().numpy())

#     avg_val_loss = val_loss / len(val_loader)
#     val_accuracy = 100 * correct / total
#     val_logloss = log_loss(all_labels, all_probs, labels=list(range(len(class_names))))

#     # 결과 출력
#     print(f"Train Loss : {avg_train_loss:.4f} || Valid Loss : {avg_val_loss:.4f} | Valid Accuracy : {val_accuracy:.4f}%")

#     # Best model 저장
#     if val_logloss < best_logloss:
#         best_logloss = val_logloss
#         torch.save(model.state_dict(), f'best_model.pth')
#         print(f"📦 Best model saved at epoch {epoch+1} (logloss: {val_logloss:.4f})")

from sklearn.metrics import log_loss
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import classification_report
import os

# ✅ 모델, 손실, 옵티마이저, 스케줄러
model = BaseModel(num_classes=len(class_names)).to(device)
best_logloss = float('inf')
start_epoch = 0

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(model.parameters(), lr=CFG['LEARNING_RATE'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['EPOCHS'])
scaler = GradScaler()

# ✅ 체크포인트 재개 기능 (중간에 끊겼을 때 대비)
checkpoint_path = 'checkpoint_latest.pth'
if os.path.exists(checkpoint_path):
    print("🔄 이전 체크포인트 불러오는 중...")
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model'])
    optimizer.load_state_dict(checkpoint['optimizer'])
    scheduler.load_state_dict(checkpoint['scheduler'])
    scaler.load_state_dict(checkpoint['scaler'])
    start_epoch = checkpoint['epoch'] + 1
    best_logloss = checkpoint['best_logloss']
    print(f"✅ {start_epoch} 에폭부터 재개합니다.")

# ✅ 학습 루프
for epoch in range(start_epoch, CFG['EPOCHS']):
    print(f"\n📘 Epoch {epoch+1}/{CFG['EPOCHS']}")
    
    model.train()
    train_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"[Epoch {epoch+1}] Training"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # ✅ 검증 루프
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"[Epoch {epoch+1}] Validation"):
            images, labels = images.to(device), labels.to(device)

            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            val_loss += loss.item()

            # Accuracy
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            probs = F.softmax(outputs, dim=1)
            probs = torch.clamp(probs, 1e-7, 1 - 1e-7)
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_accuracy = 100 * correct / total
    val_logloss = log_loss(all_labels, all_probs, labels=list(range(len(class_names))))

    print(f"✅ Train Loss : {avg_train_loss:.4f} | Valid Loss : {avg_val_loss:.4f} | Valid Acc : {val_accuracy:.2f}% | LogLoss : {val_logloss:.4f}")

    # ✅ Best 모델 저장
    if val_logloss < best_logloss:
        best_logloss = val_logloss
        torch.save(model.state_dict(), f'best_model.pth')
        print(f"💾 Best model saved at epoch {epoch+1} (logloss: {val_logloss:.4f})")

    # ✅ 현재 체크포인트 저장
    torch.save({
        'epoch': epoch,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler': scaler.state_dict(),
        'best_logloss': best_logloss
    }, checkpoint_path)

    scheduler.step()

/tmp/ipykernel_35/226424495.py:77: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()



📘 Epoch 1/50


[Epoch 1] Training:   0%|          | 0/829 [00:00<?, ?it/s]/tmp/ipykernel_35/226424495.py:102: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
[Epoch 1] Validation:   0%|          | 0/208 [00:00<?, ?it/s]/tmp/ipykernel_35/226424495.py:125: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
[Epoch 1] Validation: 100%|██████████| 208/208 [00:32<00:00,  6.41it/s]


✅ Train Loss : 5.9247 | Valid Loss : 5.6478 | Valid Acc : 1.78% | LogLoss : 5.5475
💾 Best model saved at epoch 1 (logloss: 5.5475)

📘 Epoch 2/50


[Epoch 2] Training:   0%|          | 0/829 [00:00<?, ?it/s]/tmp/ipykernel_35/226424495.py:102: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
[Epoch 2] Validation:   0%|          | 0/208 [00:00<?, ?it/s]/tmp/ipykernel_35/226424495.py:125: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
[Epoch 2] Validation: 100%|██████████| 208/208 [00:20<00:00, 10.03it/s]


...

✅ Train Loss : 1.1057 | Valid Loss : 1.1923 | Valid Acc : 94.81% | LogLoss : 0.2877

📘 Epoch 49/50


[Epoch 49] Training:   0%|          | 0/829 [00:00<?, ?it/s]/tmp/ipykernel_35/226424495.py:102: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
[Epoch 49] Validation:   0%|          | 0/208 [00:00<?, ?it/s]/tmp/ipykernel_35/226424495.py:125: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
[Epoch 49] Validation: 100%|██████████| 208/208 [00:20<00:00, 10.28it/s]


✅ Train Loss : 1.1032 | Valid Loss : 1.1922 | Valid Acc : 94.75% | LogLoss : 0.2946

📘 Epoch 50/50


[Epoch 50] Training:   0%|          | 0/829 [00:00<?, ?it/s]/tmp/ipykernel_35/226424495.py:102: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
[Epoch 50] Validation:   0%|          | 0/208 [00:00<?, ?it/s]/tmp/ipykernel_35/226424495.py:125: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
[Epoch 50] Validation: 100%|██████████| 208/208 [00:20<00:00, 10.40it/s]


✅ Train Loss : 1.1047 | Valid Loss : 1.1893 | Valid Acc : 94.78% | LogLoss : 0.2979


# Inference

In [14]:
test_dataset = CustomImageDataset(test_root, transform=val_transform, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False)

In [15]:
# 저장된 모델 로드
model = BaseModel(num_classes=len(class_names))
model.load_state_dict(torch.load('best_model.pth', map_location=device))
model.to(device)

# 추론
model.eval()
results = []

with torch.no_grad():
    for images in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = F.softmax(outputs, dim=1)

        # 각 배치의 확률을 리스트로 변환
        for prob in probs.cpu():  # prob: (num_classes,)
            result = {
                class_names[i]: prob[i].item()
                for i in range(len(class_names))
            }
            results.append(result)
            
pred = pd.DataFrame(results)

# Submission

In [22]:
submission = pd.read_csv('/kaggle/input/sample_submission.csv', encoding='utf-8-sig')

# 'ID' 컬럼을 제외한 클래스 컬럼 정렬
class_columns = submission.columns[1:]
pred = pred[class_columns]

submission[class_columns] = pred.values
submission.to_csv('submission.csv', index=False, encoding='utf-8-sig')